# RoPE deep dive: from a rotated vector to a changed attention score

You just hand-verified two rotations from the worked example in
[`journey_of_a_model/003-norm-and-positional.md`](../journey_of_a_model/003-norm-and-positional.md):

- **Pair 0** $(q_0,q_2)=(1,1)$ rotated by $2$ radians $\rightarrow (-1.3254,\ 0.4932)$
- **Pair 1** $(q_1,q_3)=(1,0)$ rotated by $0.02$ radians $\rightarrow (0.9998,\ 0.0200)$

Both matched the project's real `RotaryEmbedding` implementation exactly. This
notebook turns that hand-checked algebra into pictures, then follows the idea
all the way through to a real attention score, using **only this repo's own
real code** (`qwenmoe/model/positional.py::RotaryEmbedding`) and, in the
second half, **real trained checkpoints** -- never fabricated numbers,
matching this project's own documentation convention.

## Plan
1. **Part 1 -- the vector.** Reproduce the worked example with real code, then
   visualize what "rotate each pair by its own frequency" actually looks
   like: first as a single rotation, then across many positions, then scaled
   up to the model's real `head_dim`.
2. **Part 2 -- the attention score.** Show *why* rotating Q and K before the
   dot product is useful: the resulting score depends only on the
   **relative** distance between two tokens, never their absolute positions.
3. **Part 3 -- the real model.** Pull real pre-RoPE Q/K out of a real
   trained checkpoint and watch the real attention pattern change once RoPE
   is switched on -- then compare against an independently-trained NoPE
   checkpoint of the same architecture.

In [203]:
from pathlib import Path
import os

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

import math

import numpy as np
import torch
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots

from qwenmoe.model.positional import RotaryEmbedding

print("torch version:", torch.__version__)

Working directory: /Users/sjoseph/Documents/Study/training-my-moe
torch version: 2.13.0


## Part 1 -- the vector

### Reproducing the worked example with real code

Same numbers as the journal entry: `head_dim=4`, `theta=10000`, position
$m=2$, $q=[1,1,1,0]$. If the real `RotaryEmbedding` class produces the same
rotated vector the hand derivation did, everything plotted below is showing
the real algorithm, not a simplified stand-in.

In [204]:
rope_toy = RotaryEmbedding(head_dim=4, theta=10000.0)
print("inv_freq:", [round(v, 4) for v in rope_toy.inv_freq.tolist()])

position = torch.tensor([2.0])
q = torch.tensor([[1.0, 1.0, 1.0, 0.0]])  # (seq_len=1, head_dim=4)
q_rot, _ = rope_toy.apply_to_qk(q, q, position)

print("angles at position 2:", [round(v, 4) for v in rope_toy.positional_trace(position)[0].tolist()])
print("q before rotation:", q[0].tolist())
print("q after rotation: ", [round(v, 4) for v in q_rot[0].tolist()])

expected = torch.tensor([-1.3254, 0.9998, 0.4932, 0.0200])
assert torch.allclose(q_rot[0], expected, atol=1e-3)
print("\nMatches the hand-verified worked example exactly.")

inv_freq: [1.0, 0.01]
angles at position 2: [2.0, 0.02, 2.0, 0.02]
q before rotation: [1.0, 1.0, 1.0, 0.0]
q after rotation:  [-1.3254, 0.9998, 0.4932, 0.02]

Matches the hand-verified worked example exactly.


### Graph 1 -- a rotation, drawn

Pair 0 (indices 0 and 2) starts at $(1,1)$ and rotates by $2$ radians; pair 1
(indices 1 and 3) starts at $(1,0)$ and rotates by only $0.02$ radians --
different starting points, but literally the same rotation-matrix formula,
just with a different frequency plugged in. The gray arc traces the sweep
from the blue (before) arrow to the red (after) arrow.

In [205]:
def rotate2d(vec, theta_rad):
    c, s = math.cos(theta_rad), math.sin(theta_rad)
    rotation = np.array([[c, -s], [s, c]])
    return rotation @ np.array(vec)

assert np.allclose(rotate2d([1.0, 1.0], 2.0), [-1.3254, 0.4932], atol=1e-3)
assert np.allclose(rotate2d([1.0, 0.0], 0.02), [0.9998, 0.0200], atol=1e-3)

specs = [((1.0, 1.0), 2.0, "pair 0 = (q0, q2): angle = 2 rad (fast)"),
         ((1.0, 0.0), 0.02, "pair 1 = (q1, q3): angle = 0.02 rad (slow)")]
fig1 = make_subplots(rows=1, cols=2, subplot_titles=[s[2] for s in specs])
for col, (before_vec, angle, _) in enumerate(specs, start=1):
    before = np.array(before_vec)
    after = rotate2d(before_vec, angle)
    r_circle = math.hypot(*before)
    theta_circle = np.linspace(0, 2 * math.pi, 100)
    start = math.atan2(before[1], before[0])
    arc = np.linspace(start, start + angle, 50)
    r_arc = 0.5

    fig1.add_trace(go.Scatter(x=r_circle * np.cos(theta_circle), y=r_circle * np.sin(theta_circle),
                               mode="lines", line=dict(color="lightgray", dash="dot"),
                               showlegend=False, hoverinfo="skip"), row=1, col=col)
    fig1.add_trace(go.Scatter(x=r_arc * np.cos(arc), y=r_arc * np.sin(arc), mode="lines",
                               line=dict(color="gray", width=2), showlegend=False, hoverinfo="skip"),
                   row=1, col=col)
    fig1.add_trace(go.Scatter(x=[0, before[0]], y=[0, before[1]], mode="lines+markers",
                               line=dict(color="royalblue", width=3), marker=dict(size=9),
                               name=f"before ({before[0]:.3f}, {before[1]:.3f})"), row=1, col=col)
    fig1.add_trace(go.Scatter(x=[0, after[0]], y=[0, after[1]], mode="lines+markers",
                               line=dict(color="firebrick", width=3), marker=dict(size=9),
                               name=f"after  ({after[0]:.3f}, {after[1]:.3f})"), row=1, col=col)
    xaxis_name = "x" if col == 1 else f"x{col}"
    fig1.update_xaxes(range=[-1.6, 1.6], zeroline=True, row=1, col=col)
    fig1.update_yaxes(range=[-1.6, 1.6], zeroline=True, scaleanchor=xaxis_name, scaleratio=1, row=1, col=col)

fig1.update_layout(title="The rotation formula applied to both pairs of the worked example", width=880, height=460)
fig1.show()

### Graph 2 -- fast dial vs slow dial across many positions

Same 4-dim vector $[1,1,1,0]$ as the worked example -- pair 0's slice is
$(1,1)$, pair 1's slice is $(1,0)$. Press **Play** (or drag the slider) to
sweep the position $m$ from 0 to 40 and watch both pairs rotate. Pair 0's
angle grows by exactly 1 radian per position, so 40 positions sweep past
**more than 6 full turns** -- it laps the circle several times. Pair 1's
angle grows by only 0.01 radian per position, so the *same* 40 positions
barely nudge it. This is the "clock hands moving at different speeds"
analogy from the journal entry, made concrete -- the dotted ghost trail
shows each pair's full path across the sweep.

In [206]:
positions = torch.arange(0, 41, dtype=torch.float32)
q_content = torch.tensor([1.0, 1.0, 1.0, 0.0]).repeat(len(positions), 1)  # same content at every position
q_rot_all, _ = rope_toy.apply_to_qk(q_content, q_content, positions)
q_rot_all = q_rot_all.numpy()

pair2_x = [q_rot_all[:, 0], q_rot_all[:, 1]]
pair2_y = [q_rot_all[:, 2], q_rot_all[:, 3]]
pair2_names = ["pair 0 (freq=1.0)", "pair 1 (freq=0.01)"]
pair2_colors = pc.sample_colorscale("Viridis", [0, 1])

ghost2 = [
    go.Scatter(x=pair2_x[i], y=pair2_y[i], mode="lines", line=dict(color=pair2_colors[i], width=1, dash="dot"),
               opacity=0.4, showlegend=False, hoverinfo="skip")
    for i in range(2)
]
anim2 = [
    go.Scatter(x=[0, pair2_x[i][0]], y=[0, pair2_y[i][0]], mode="lines+markers",
               line=dict(color=pair2_colors[i], width=3), marker=dict(size=9, color=pair2_colors[i]),
               name=pair2_names[i])
    for i in range(2)
]
frames2 = [
    go.Frame(name=str(int(m)),
             data=[go.Scatter(x=[0, pair2_x[i][k]], y=[0, pair2_y[i][k]]) for i in range(2)],
             traces=[2, 3])
    for k, m in enumerate(positions.tolist())
]

fig2 = go.Figure(data=ghost2 + anim2, frames=frames2)
fig2.update_layout(
    title="Same 4-dim vector [1,1,1,0], rotated by position x frequency per pair",
    xaxis=dict(range=[-1.6, 1.6], zeroline=True, title="x"),
    yaxis=dict(range=[-1.6, 1.6], zeroline=True, title="y", scaleanchor="x", scaleratio=1),
    width=760, height=620,
    updatemenus=[dict(type="buttons", showactive=False, x=0.05, y=1.08, buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=140, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(steps=[
        dict(method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True))], label=f.name)
        for f in frames2
    ], currentvalue=dict(prefix="position m = "))],
)
fig2.show()

### Graph 3 -- scaling up to the real model's head_dim

The toy example only had 2 frequencies. The real `SMOKE_TEST` config uses
`d_model=64, n_heads=4` -> `head_dim=16` -> **8** frequencies, geometrically
spaced from `inv_freq[0]=1.0` down to a tiny value. Left: the frequency
spectrum (log scale). Right: `cos(angle)` for every (position, frequency)
pair over a 32-token window -- the standard "RoPE fingerprint" picture. Fast
rows (like the toy example's pair 0) oscillate visibly; slow rows near the
bottom barely change over this whole window.

In [207]:
from configs.presets import SMOKE_TEST

head_dim = SMOKE_TEST.d_model // SMOKE_TEST.n_heads
print(f"SMOKE_TEST head_dim = {head_dim}  (d_model={SMOKE_TEST.d_model} / n_heads={SMOKE_TEST.n_heads})")

rope_real = RotaryEmbedding(head_dim=head_dim, theta=SMOKE_TEST.rope_theta)
n_freqs = head_dim // 2

seq_len = 32
positions = torch.arange(seq_len, dtype=torch.float32)
angles = rope_real.positional_trace(positions)[:, :n_freqs]  # (seq_len, n_freqs)

fig3 = make_subplots(rows=1, cols=2, column_widths=[0.35, 0.65], subplot_titles=[
    f"{n_freqs} frequencies for head_dim={head_dim}", "cos(m x inv_freq[i])",
])
fig3.add_trace(go.Scatter(x=list(range(n_freqs)), y=rope_real.inv_freq.numpy(), mode="lines+markers",
                           showlegend=False), row=1, col=1)
fig3.update_yaxes(type="log", title="inv_freq[i]  (log scale)", row=1, col=1)
fig3.update_xaxes(title="dim-pair index i", row=1, col=1)

fig3.add_trace(go.Heatmap(z=angles.cos().numpy().T, colorscale="RdBu", zmin=-1, zmax=1,
                           colorbar=dict(title="cos(angle)")), row=1, col=2)
fig3.update_xaxes(title="position m", row=1, col=2)
fig3.update_yaxes(title="dim-pair index i (0=fastest)", row=1, col=2)

fig3.update_layout(title=f"RoPE frequency spectrum and rotation-angle fingerprint (head_dim={head_dim})",
                    width=980, height=420)
fig3.show()

period_fastest = 2 * math.pi / rope_real.inv_freq[0].item()
period_slowest = 2 * math.pi / rope_real.inv_freq[-1].item()
print(f"pair 0 completes a full revolution every {period_fastest:.2f} positions")
print(f"the slowest pair needs {period_slowest:,.0f} positions for one revolution")
print(f"(SMOKE_TEST's max_seq_len is only {SMOKE_TEST.max_seq_len} -- that pair never even completes one full turn)")

SMOKE_TEST head_dim = 16  (d_model=64 / n_heads=4)


pair 0 completes a full revolution every 6.28 positions
the slowest pair needs 19,869 positions for one revolution
(SMOKE_TEST's max_seq_len is only 128 -- that pair never even completes one full turn)


### Graph 4 -- rotation never changes length

A rotation matrix is length-preserving by definition -- $\|R(\theta)v\| =
\|v\|$ for any angle $\theta$. That means RoPE can inject a position signal
into $q$/$k$ **without** touching the vector's magnitude, which is what the
rest of the network (dot products, softmax) is sensitive to. Verified here
across 200 positions on a random real-sized (`head_dim=16`) vector, not just
the tiny 4-dim toy example.

In [208]:
torch.manual_seed(0)
vec = torch.randn(1, head_dim)
positions_wide = torch.arange(0, 200, dtype=torch.float32)
vec_rep = vec.repeat(len(positions_wide), 1)
vec_rot, _ = rope_real.apply_to_qk(vec_rep, vec_rep, positions_wide)

norm_before = vec_rep.norm(dim=-1)
norm_after = vec_rot.norm(dim=-1)

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=positions_wide.numpy(), y=norm_before.numpy(), mode="lines",
                           line=dict(color="royalblue", width=6), opacity=0.4, name="before rotation"))
fig4.add_trace(go.Scatter(x=positions_wide.numpy(), y=norm_after.numpy(), mode="lines",
                           line=dict(color="firebrick", width=2, dash="dash"), name="after rotation"))
fig4.update_layout(title="Rotation never changes the vector's length, at any position",
                    xaxis_title="position m", yaxis_title="||vector||",
                    yaxis=dict(range=[0, norm_before.max().item() * 1.3]),
                    width=760, height=340)
fig4.show()

print("max |norm_after - norm_before| across all 200 positions:", (norm_after - norm_before).abs().max().item())

max |norm_after - norm_before| across all 200 positions: 2.384185791015625e-07


## Part 2 -- from vectors to attention scores

An attention score is just a dot product, $q \cdot k / \sqrt{\text{head\_dim}}$.
**Without any positional information**, that score only reflects *content*
similarity -- shuffle the token order and every score stays exactly the
same, since positions never entered the computation at all. RoPE fixes this
by rotating $q$ at position $m$ and $k$ at position $n$ *before* the dot
product. The journal entry's complex-number derivation proves this
algebraically: rotating both sides turns the score into a function of
$(m-n)$ only, never $m$ and $n$ individually.

### Graph 5 -- same relative distance, same score

To check this isn't just an algebraic curiosity: fix 5 tokens' content
vectors, score them at positions $[0,1,2,3,4]$, then score the *exact same
content* again at positions $[50,51,52,53,54]$ -- same spacing, shifted by
50. If the relative-position claim is right, both score matrices should be
numerically identical.

In [209]:
torch.manual_seed(42)
n_tokens = 5
q_content = torch.randn(n_tokens, head_dim)
k_content = torch.randn(n_tokens, head_dim)
scale = math.sqrt(head_dim)

def rope_scores(positions):
    q_r, k_r = rope_real.apply_to_qk(q_content, k_content, positions)
    return (q_r @ k_r.T) / scale

positions_a = torch.arange(n_tokens, dtype=torch.float32)   # [0, 1, 2, 3, 4]
positions_b = positions_a + 50                              # [50, 51, 52, 53, 54] -- same spacing

scores_raw = (q_content @ k_content.T) / scale               # no positional info at all
scores_a = rope_scores(positions_a)
scores_b = rope_scores(positions_b)

max_diff = (scores_a - scores_b).abs().max().item()
print(f"max|scores(pos 0-4) - scores(pos 50-54)| = {max_diff:.2e}  (same spacing -> same scores)")
assert max_diff < 1e-4

titles = ["no positional info at all (content-only)", "RoPE @ positions 0..4",
          "RoPE @ positions 50..54 (identical to the middle panel)"]
fig5 = make_subplots(rows=1, cols=3, subplot_titles=titles)
for col, mat in enumerate([scores_raw, scores_a, scores_b], start=1):
    fig5.add_trace(go.Heatmap(z=mat.detach().numpy(), zmin=-3, zmax=3, zmid=0, colorscale="RdBu",
                               showscale=(col == 3),
                               colorbar=dict(title="q.k/sqrt(head_dim)") if col == 3 else None),
                   row=1, col=col)
    fig5.update_xaxes(title="key token", row=1, col=col)
    fig5.update_yaxes(title="query token", autorange="reversed", row=1, col=col)
fig5.update_layout(title="Same relative distance -> same score", width=1200, height=420)
fig5.show()

max|scores(pos 0-4) - scores(pos 50-54)| = 4.77e-07  (same spacing -> same scores)


## Part 3 -- tying it back to the real trained model

Everything so far used synthetic vectors. Now pull real pre-RoPE Q/K out of
the real `runs/smoke_test/checkpoints/step_1000.pt` checkpoint for the same
`"ROMEO:"` prompt used throughout this project's journal and
`site/token-journey/` (SMOKE_TEST config: `d_model=64, n_heads=4, n_kv_heads=2`
-> `head_dim=16`).

In [210]:
from qwenmoe.model.transformer import QwenMoEModel
from qwenmoe.tokenizer.base import build_tokenizer_from_meta
from qwenmoe.train.trainer import load_checkpoint


def load_model_and_trace(checkpoint_path, prompt):
    checkpoint = load_checkpoint(checkpoint_path)
    model = QwenMoEModel(checkpoint["model_config"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    tokenizer = build_tokenizer_from_meta(checkpoint["tokenizer_meta"])
    token_ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([token_ids], dtype=torch.long)
    with torch.no_grad():
        _, bundle = model.forward_with_trace(input_ids)
    return model, tokenizer, bundle


model, tokenizer, bundle = load_model_and_trace("runs/smoke_test/checkpoints/step_1000.pt", "ROMEO:")
tokens = [tokenizer.decode([tid]) for tid in bundle.token_ids]
real_head_dim = model.config.d_model // model.config.n_heads
print(f"checkpoint config: d_model={model.config.d_model} n_heads={model.config.n_heads} "
      f"n_kv_heads={model.config.n_kv_heads} head_dim={real_head_dim}")
print("tokens:", tokens)

checkpoint config: d_model=64 n_heads=4 n_kv_heads=2 head_dim=16
tokens: ['R', 'O', 'M', 'E', 'O', ':']


### Graph 6 -- the real model's attention scores, with vs without RoPE

`LayerTrace.q`/`.k` are captured **before** RoPE is applied (exactly so this
kind of comparison is possible). Left panel: what the dot product of these
*exact real learned* Q/K vectors would be if the RoPE step were skipped (a
counterfactual -- same weights, just skip the rotation). Right panel: the
real computation, RoPE applied via this checkpoint's own `RotaryEmbedding`
module -- verified below to match the trace this checkpoint actually
recorded during its real forward pass.

In [211]:
layer0 = bundle.layers[0]
n_heads, seq_len, _ = layer0.q.shape
n_kv_heads = layer0.k.shape[0]
n_rep = n_heads // n_kv_heads
positions = torch.arange(seq_len, dtype=torch.float32)
scale = math.sqrt(real_head_dim)

# counterfactual: same real learned Q/K, RoPE step skipped
k_rep_no_rope = layer0.k.repeat_interleave(n_rep, dim=0)
scores_no_rope = (layer0.q @ k_rep_no_rope.transpose(-2, -1)) / scale

# real: this checkpoint's own RotaryEmbedding module, applied before the dot product
positional = model.blocks[0].attn.positional
q_rot, k_rot = positional.apply_to_qk(layer0.q, layer0.k, positions)
k_rot_rep = k_rot.repeat_interleave(n_rep, dim=0)
scores_rope = (q_rot @ k_rot_rep.transpose(-2, -1)) / scale

max_diff = (scores_rope - layer0.attn_scores_pre_mask).abs().max().item()
print("max diff vs the real captured attn_scores_pre_mask:", max_diff)
assert max_diff < 1e-4

head = 0
titles = ["counterfactual: RoPE skipped (same real learned Q/K)",
          "real: RoPE applied (matches this checkpoint exactly)"]
fig6 = make_subplots(rows=1, cols=2, subplot_titles=titles)
for col, mat in enumerate([scores_no_rope[head], scores_rope[head]], start=1):
    fig6.add_trace(go.Heatmap(z=mat.detach().numpy(), x=tokens, y=tokens, colorscale="RdBu", zmid=0,
                               showscale=(col == 2),
                               colorbar=dict(title="q.k/sqrt(head_dim)") if col == 2 else None),
                   row=1, col=col)
    fig6.update_xaxes(title="key token", row=1, col=col)
    fig6.update_yaxes(title="query token", autorange="reversed", row=1, col=col)
fig6.update_layout(title=f'head {head}, real SMOKE_TEST checkpoint, prompt "ROMEO:"', width=820, height=420)
fig6.show()

max diff vs the real captured attn_scores_pre_mask: 0.0


### Graph 7 -- does this actually matter when you train two real models?

The comparison above holds one model's weights fixed. As a final, fully-real
check: `runs/ablation_study/` already contains two **independently trained**
checkpoints that differ in exactly one axis -- `SMOKE_TEST`
(`positional_scheme="rope"`) and `POS_NOPE` (`positional_scheme="nope"`) --
same architecture, same data, same training steps. Real final
(post-causal-mask, post-softmax) attention weights for the same prompt, from
two genuinely different trained models.

In [212]:
model_rope, tok_rope, bundle_rope = load_model_and_trace(
    "runs/ablation_study/SMOKE_TEST/checkpoints/step_600.pt", "ROMEO:"
)
model_nope, tok_nope, bundle_nope = load_model_and_trace(
    "runs/ablation_study/POS_NOPE/checkpoints/step_600.pt", "ROMEO:"
)
tokens_ablation = [tok_rope.decode([tid]) for tid in bundle_rope.token_ids]
assert tokens_ablation == [tok_nope.decode([tid]) for tid in bundle_nope.token_ids]

attn_rope = bundle_rope.layers[0].attn_scores_post_softmax[0].detach().numpy()
attn_nope = bundle_nope.layers[0].attn_scores_post_softmax[0].detach().numpy()

fig7 = make_subplots(rows=1, cols=2, subplot_titles=["trained WITH RoPE", "trained WITHOUT RoPE (NoPE)"])
for col, mat in enumerate([attn_rope, attn_nope], start=1):
    fig7.add_trace(go.Heatmap(z=mat, x=tokens_ablation, y=tokens_ablation, colorscale="Viridis",
                               zmin=0, zmax=1, showscale=(col == 2),
                               colorbar=dict(title="attention weight") if col == 2 else None),
                   row=1, col=col)
    fig7.update_xaxes(title="key token", row=1, col=col)
    fig7.update_yaxes(title="query token", autorange="reversed", row=1, col=col)
fig7.update_layout(title='real, independently-trained checkpoints -- head 0, layer 0, prompt "ROMEO:"',
                    width=820, height=420)
fig7.show()

last_token_row = -1
print("last token (':') attention over the prompt:")
print("  with RoPE:   ", np.round(attn_rope[last_token_row], 3))
print("  without RoPE:", np.round(attn_nope[last_token_row], 3))

last token (':') attention over the prompt:
  with RoPE:    [0.127 0.11  0.174 0.279 0.208 0.103]
  without RoPE: [0.199 0.132 0.21  0.133 0.132 0.194]


## Summary

- A **rotation matrix** applied to a 2D pair of a query/key vector is the
  entire mechanical idea behind RoPE -- verified by hand in the journal
  entry and reproduced here with the project's real code (Part 1, Graph 1).
- Splitting a `head_dim`-sized vector into pairs and giving each pair its
  own frequency (`inv_freq[i] = theta^(-2i/head_dim)`) means some pairs spin
  fast (fine position resolution) and others spin almost imperceptibly
  slowly (coarse, long-range resolution) -- Graphs 2-3.
- Rotation is length-preserving, so RoPE injects *where* a token is without
  ever distorting *what* the vector already encoded -- Graph 4.
- Because rotating both $q$ and $k$ before the dot product is algebraically
  equivalent to rotating by their *difference*, the resulting attention
  score is provably a function of relative position only -- proven both
  algebraically (journal entry) and numerically here (Graph 5).
- The same effect shows up in a real trained checkpoint: skipping the RoPE
  step on the exact same learned Q/K changes the real attention scores
  (Graph 6), and two independently-trained models (RoPE vs NoPE) end up with
  visibly different real attention patterns for the same prompt (Graph 7).

**Further reading:** full algebraic derivation and the original worked
example -- [`journey_of_a_model/003-norm-and-positional.md`](../journey_of_a_model/003-norm-and-positional.md).
Interactive, per-token walkthrough of a real forward pass (including a
zoomable RoPE step) -- [`site/token-journey/index.html`](../site/token-journey/index.html).

## Extra: the same worked example, scaled up to 16 dimensions (8 pairs)

Part 1's worked example used a 4-dimensional toy vector (2 pairs) so the
rotation could be checked by hand. The exact same `RotaryEmbedding` code
works identically for any even `head_dim` -- below it runs again with
`head_dim=16` (8 pairs, the model's *real* size from Part 3), letting the
code do every rotation instead of deriving it by hand. Every pair starts at
the same point $(1,1)$, so the only thing that differs between them is
frequency.

In [213]:
rope16 = RotaryEmbedding(head_dim=16, theta=10000.0)
print("head_dim=16 -> 8 pairs (vs. the 4-dim / 2-pair toy example in Part 1)")
print("inv_freq:", [round(v, 6) for v in rope16.inv_freq.tolist()])

position16 = torch.tensor([2.0])
q16_single = torch.ones(1, 16)  # every pair starts at (1, 1), same idea as the 4-dim example
q16_rot, _ = rope16.apply_to_qk(q16_single, q16_single, position16)
angles16 = rope16.positional_trace(position16)[0, :8]

print(f"\nall 8 pairs rotated by position m=2, same position as the 4-dim example:\n")
for i in range(8):
    before_pair = (q16_single[0, i].item(), q16_single[0, i + 8].item())
    after_pair = (round(q16_rot[0, i].item(), 4), round(q16_rot[0, i + 8].item(), 4))
    print(f"  pair {i}: freq={rope16.inv_freq[i].item():.6f}  angle={angles16[i].item():.4f} rad   "
          f"{before_pair} -> {after_pair}")

# cross-check pair 0 against the plain 2D rotation matrix, same check as the 4-dim example
manual_pair0 = rotate2d([1.0, 1.0], angles16[0].item())
assert np.allclose(manual_pair0, [q16_rot[0, 0].item(), q16_rot[0, 8].item()], atol=1e-4)
print("\npair 0 matches the plain rotation-matrix formula exactly, same as the 4-dim example.")

head_dim=16 -> 8 pairs (vs. the 4-dim / 2-pair toy example in Part 1)
inv_freq: [1.0, 0.316228, 0.1, 0.031623, 0.01, 0.003162, 0.001, 0.000316]

all 8 pairs rotated by position m=2, same position as the 4-dim example:

  pair 0: freq=1.000000  angle=2.0000 rad   (1.0, 1.0) -> (-1.3254, 0.4932)
  pair 1: freq=0.316228  angle=0.6325 rad   (1.0, 1.0) -> (0.2155, 1.3977)
  pair 2: freq=0.100000  angle=0.2000 rad   (1.0, 1.0) -> (0.7814, 1.1787)
  pair 3: freq=0.031623  angle=0.0632 rad   (1.0, 1.0) -> (0.9348, 1.0612)
  pair 4: freq=0.010000  angle=0.0200 rad   (1.0, 1.0) -> (0.9798, 1.0198)
  pair 5: freq=0.003162  angle=0.0063 rad   (1.0, 1.0) -> (0.9937, 1.0063)
  pair 6: freq=0.001000  angle=0.0020 rad   (1.0, 1.0) -> (0.998, 1.002)
  pair 7: freq=0.000316  angle=0.0006 rad   (1.0, 1.0) -> (0.9994, 1.0006)

pair 0 matches the plain rotation-matrix formula exactly, same as the 4-dim example.


### An interactive, animated view of all 8 pairs at once

A single static snapshot doesn't scale well past 2 pairs, so this one adds
real animation: press **Play** (or drag the slider) to sweep the position
$m$ from 0 to 40 and watch all 8 pairs rotate together -- pair 0 (fastest)
laps the circle several times while pair 7 (slowest) barely creeps off its
starting point. The dotted "ghost" trail behind each arrow is that pair's
full path across every position in the sweep, so you can see both *where
it is right now* (the solid arrow) and *where it has been* (the trail) at
the same time. Hover any arrow to read its exact frequency, and click a
legend entry to hide/show that pair.

In [214]:
n_pairs = 8
positions_anim = torch.arange(0, 41, dtype=torch.float32)
q16 = torch.ones(len(positions_anim), 16)  # same (1,1)-per-pair content at every position
q16_rot_all, _ = rope16.apply_to_qk(q16, q16, positions_anim)
q16_rot_all = q16_rot_all.numpy()  # (41, 16)

pair_x = q16_rot_all[:, :n_pairs]   # (41, 8)
pair_y = q16_rot_all[:, n_pairs:]   # (41, 8)
freqs16 = rope16.inv_freq.numpy()
colors = pc.sample_colorscale("Viridis", [i / (n_pairs - 1) for i in range(n_pairs)])

ghost_traces = [
    go.Scatter(
        x=pair_x[:, i], y=pair_y[:, i], mode="lines",
        line=dict(color=colors[i], width=1, dash="dot"),
        opacity=0.35, showlegend=False, hoverinfo="skip",
    )
    for i in range(n_pairs)
]
anim_traces = [
    go.Scatter(
        x=[0, pair_x[0, i]], y=[0, pair_y[0, i]], mode="lines+markers",
        line=dict(color=colors[i], width=3), marker=dict(size=9, color=colors[i]),
        name=f"pair {i} (freq={freqs16[i]:.4f})",
    )
    for i in range(n_pairs)
]

frames = [
    go.Frame(
        name=str(int(m)),
        data=[go.Scatter(x=[0, pair_x[k, i]], y=[0, pair_y[k, i]]) for i in range(n_pairs)],
        traces=list(range(n_pairs, 2 * n_pairs)),
    )
    for k, m in enumerate(positions_anim.tolist())
]

fig = go.Figure(data=ghost_traces + anim_traces, frames=frames)
fig.update_layout(
    title="16-dim vector (8 pairs), all starting at (1,1) -- rotated by position x each pair's own frequency",
    xaxis=dict(range=[-1.6, 1.6], zeroline=True, title="x"),
    yaxis=dict(range=[-1.6, 1.6], zeroline=True, title="y", scaleanchor="x", scaleratio=1),
    width=780, height=650,
    updatemenus=[dict(
        type="buttons", showactive=False, x=0.05, y=1.08,
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=180, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
        ],
    )],
    sliders=[dict(
        steps=[
            dict(method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True))], label=f.name)
            for f in frames
        ],
        currentvalue=dict(prefix="position m = "),
    )],
)
fig.show()

### The actual vector, spelled out -- position by position

The animation above only draws the 8 pairs as arrows on a circle. This cell
prints the literal 16 real numbers RoPE actually produces -- exactly what a
model's `q` (or `k`) vector for one token would contain after rotation --
at each of the first 9 token positions (0 through 8), reusing the exact
same `rope16`/`q16_rot_all` this section already computed above (no
recomputation -- same `head_dim=16`, same all-ones-per-pair identity
content). Position 0 rotates every pair by 0 radians, so its row is just
the raw, untouched content; from position 1 onward, watch column `q0`
(pair 0, fastest, period ~6.3 positions) swing wildly row to row while
column `q15` (pair 7, slowest) barely creeps away from `1.000`.

As a cross-check against the hand-worked derivation for position $m=5$:
pair 0 $(1,1)\to(1.243,-0.675)$, pair 2 $(1,1)\to(0.398,1.357)$, pair 7
$(1,1)\to(0.998,1.002)$ -- verified exactly below.


In [215]:
n_positions_show = 9  # positions 0..8 -- covers the m=5 example from the hand derivation above
q16_vec_rot = q16_rot_all[:n_positions_show]  # reuse the animation's own real numbers, no recompute
row_labels16 = [f"pos {k}" for k in range(n_positions_show)]

print("before RoPE (every position starts here -- all-ones, per-pair identity content):")
print("  [" + ", ".join(f"{v:+.3f}" for v in q16[0].tolist()) + "]")

print(f"\nafter RoPE, the real 16-number vector at each of positions 0..{n_positions_show - 1}:")
for label, row in zip(row_labels16, q16_vec_rot):
    print(f"  {label}: [" + ", ".join(f"{v:+.3f}" for v in row) + "]")

assert np.allclose(q16_vec_rot[0], q16[0].numpy(), atol=1e-6)
print("\nposition 0 rotates every pair by 0 radians -- row is identical to the before-RoPE content, confirmed.")

# cross-check against the hand-worked position m=5 derivation (pair 0, pair 2, pair 7)
pos5 = q16_vec_rot[5]
pair0_m5 = (pos5[0], pos5[8])
pair2_m5 = (pos5[2], pos5[10])
pair7_m5 = (pos5[7], pos5[15])
assert np.allclose(pair0_m5, [1.2426, -0.6753], atol=1e-3)
assert np.allclose(pair2_m5, [0.3982, 1.3570], atol=1e-3)
assert np.allclose(pair7_m5, [0.9984, 1.0016], atol=1e-3)
print("\nposition 5 hand-derivation cross-check:")
print(f"  pair 0 (fastest, freq={freqs16[0]:.4f}): (1,1) -> ({pair0_m5[0]:.4f}, {pair0_m5[1]:.4f})  matches (1.2426, -0.6753)")
print(f"  pair 2            (freq={freqs16[2]:.4f}): (1,1) -> ({pair2_m5[0]:.4f}, {pair2_m5[1]:.4f})  matches (0.3982, 1.3570)")
print(f"  pair 7 (slowest, freq={freqs16[7]:.4f}): (1,1) -> ({pair7_m5[0]:.4f}, {pair7_m5[1]:.4f})  matches (0.9984, 1.0016)")


before RoPE (every position starts here -- all-ones, per-pair identity content):
  [+1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000]

after RoPE, the real 16-number vector at each of positions 0..8:
  pos 0: [+1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000]
  pos 1: [-0.301, +0.639, +0.895, +0.968, +0.990, +0.997, +0.999, +1.000, +1.382, +1.261, +1.095, +1.031, +1.010, +1.003, +1.001, +1.000]
  pos 2: [-1.325, +0.215, +0.781, +0.935, +0.980, +0.994, +0.998, +0.999, +0.493, +1.398, +1.179, +1.061, +1.020, +1.006, +1.002, +1.001]
  pos 3: [-1.131, -0.230, +0.660, +0.901, +0.970, +0.990, +0.997, +0.999, -0.849, +1.395, +1.251, +1.090, +1.030, +1.009, +1.003, +1.001]
  pos 4: [+0.103, -0.652, +0.532, +0.866, +0.959, +0.987, +0.996, +0.999, -1.410, +1.255, +1.310, +1.118, +1.039, +1.013, +1.004, +1.001]
  pos 5: [+1.243, -1.010, +0.398

In [216]:
fig_vec16 = go.Figure(data=go.Heatmap(
    z=q16_vec_rot, x=list(range(16)), y=list(range(n_positions_show)),
    text=[[f"{v:.3f}" for v in row] for row in q16_vec_rot], texttemplate="%{text}",
    colorscale="RdBu", zmid=0, zmin=-1.6, zmax=1.6, colorbar=dict(title="value"),
))
fig_vec16.update_xaxes(title="scalar index (q0..q15)", tickmode="array",
                        tickvals=list(range(16)), ticktext=[f"q{i}" for i in range(16)])
fig_vec16.update_yaxes(title="token position", tickmode="array",
                        tickvals=list(range(n_positions_show)), ticktext=row_labels16,
                        autorange="reversed")
fig_vec16.update_layout(
    title="head_dim=16 (8 pairs), identity (1,...,1) content -- the real vector, one row per position",
    width=980, height=420,
)
fig_vec16.show()


## Extra: every scalar as a table -- 4 real tokens, head_dim=8

The clock-hand pictures above show 2D *pairs*. This section shows the same
rotation a different way: the model's raw numbers, laid out as a table,
for a tiny concrete case -- a 4-token sequence (`seq_len=4`) and
`head_dim=8` (4 pairs), so every one of the 8 scalars in the vector is
visible at once. Left: a color-coded table of the real rotated numbers,
one row per token position. Right: that same position's 4 pairs, drawn as
arrows -- the **dashed gray arrow** is the shared *before-RoPE* reference
(every pair's raw content starts at exactly $(1,1)$, no matter the
position), and the **solid colored arrows** are that pair *after* RoPE has
rotated it for whichever position is selected. Press **Play** (or drag the
slider) to move through the 4 positions -- watch the highlighted row on
the left change while the colored arrows rotate away from the gray
reference. At position 0 the colored arrows exactly overlap the gray one
(rotating by 0 radians is no rotation at all), then they visibly peel away
by different amounts per pair -- the same fast/slow-frequency idea as
Graph 2, now readable as actual numbers instead of only a picture.

In [217]:
head_dim_tbl = 8
n_pairs_tbl = 4
seq_len_tbl = 4  # "four tokens"

rope_tbl = RotaryEmbedding(head_dim=head_dim_tbl, theta=10000.0)
positions_tbl = torch.arange(seq_len_tbl, dtype=torch.float32)
q_tbl = torch.ones(seq_len_tbl, head_dim_tbl)  # every pair starts at (1,1), same convention as the 16-dim example
q_tbl_rot, _ = rope_tbl.apply_to_qk(q_tbl, q_tbl, positions_tbl)
q_tbl_rot = q_tbl_rot.numpy()  # (4, 8) -- row = token position, column = scalar q0..q7

col_labels = [f"q{i}" for i in range(head_dim_tbl)]
row_labels = [f"pos {k}" for k in range(seq_len_tbl)]

print("The vector's 8 real numbers at every one of the 4 token positions:")
for label, row in zip(row_labels, q_tbl_rot):
    print(f"  {label}: [" + ", ".join(f"{v:+.3f}" for v in row) + "]")

The vector's 8 real numbers at every one of the 4 token positions:
  pos 0: [+1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000, +1.000]
  pos 1: [-0.301, +0.895, +0.990, +0.999, +1.382, +1.095, +1.010, +1.001]
  pos 2: [-1.325, +0.781, +0.980, +0.998, +0.493, +1.179, +1.020, +1.002]
  pos 3: [-1.131, +0.660, +0.970, +0.997, -0.849, +1.251, +1.030, +1.003]


In [218]:
pair_colors_tbl = pc.sample_colorscale("Viridis", [i / (n_pairs_tbl - 1) for i in range(n_pairs_tbl)])
pair_x_tbl = q_tbl_rot[:, :n_pairs_tbl]   # (4, 4) -- x-component per pair, per position
pair_y_tbl = q_tbl_rot[:, n_pairs_tbl:]   # (4, 4) -- y-component per pair, per position
text_vals = [[f"{v:.3f}" for v in row] for row in q_tbl_rot]

fig_tbl = make_subplots(
    rows=1, cols=2, column_widths=[0.58, 0.42],
    subplot_titles=["The vector's 8 real numbers, one row per token position",
                     "That position's 4 pairs: before RoPE (gray) vs after (color)"],
)

fig_tbl.add_trace(go.Heatmap(
    z=q_tbl_rot, x=list(range(head_dim_tbl)), y=list(range(seq_len_tbl)),
    text=text_vals, texttemplate="%{text}", colorscale="RdBu", zmid=0, showscale=False,
), row=1, col=1)
fig_tbl.update_xaxes(tickmode="array", tickvals=list(range(head_dim_tbl)), ticktext=col_labels, row=1, col=1)
fig_tbl.update_yaxes(tickmode="array", tickvals=list(range(seq_len_tbl)), ticktext=row_labels,
                      autorange="reversed", row=1, col=1)

ghost_tbl = [
    go.Scatter(x=pair_x_tbl[:, i], y=pair_y_tbl[:, i], mode="lines+markers",
               line=dict(color=pair_colors_tbl[i], width=1, dash="dot"),
               marker=dict(size=4, color=pair_colors_tbl[i]),
               opacity=0.35, showlegend=False, hoverinfo="skip")
    for i in range(n_pairs_tbl)
]
before_trace = go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines+markers",
    line=dict(color="gray", width=2, dash="dash"), marker=dict(size=8, color="gray"),
    name="before RoPE (every pair starts here)",
)
anim_tbl = [
    go.Scatter(x=[0, pair_x_tbl[0, i]], y=[0, pair_y_tbl[0, i]], mode="lines+markers",
               line=dict(color=pair_colors_tbl[i], width=3), marker=dict(size=9, color=pair_colors_tbl[i]),
               name=f"after: pair {i} (q{i}, q{i + n_pairs_tbl})")
    for i in range(n_pairs_tbl)
]
for trace in ghost_tbl + [before_trace] + anim_tbl:
    fig_tbl.add_trace(trace, row=1, col=2)
fig_tbl.update_xaxes(range=[-1.6, 1.6], zeroline=True, row=1, col=2)
fig_tbl.update_yaxes(range=[-1.6, 1.6], zeroline=True, scaleanchor="x2", scaleratio=1, row=1, col=2)


def highlight_shape(k):
    return dict(type="rect", xref="x", yref="y", x0=-0.5, x1=head_dim_tbl - 0.5, y0=k - 0.5, y1=k + 0.5,
                line=dict(color="black", width=3), fillcolor="rgba(0,0,0,0)")


n_ghost_tbl = len(ghost_tbl)
before_trace_idx = 1 + n_ghost_tbl  # table=0, ghosts=1..4, before=5
arrow_trace_idx = list(range(before_trace_idx + 1, before_trace_idx + 1 + n_pairs_tbl))  # after-arrows=6..9

frames_tbl = []
for k in range(seq_len_tbl):
    frame_data = [go.Scatter(x=[0, pair_x_tbl[k, i]], y=[0, pair_y_tbl[k, i]]) for i in range(n_pairs_tbl)]
    frames_tbl.append(go.Frame(name=str(k), data=frame_data, traces=arrow_trace_idx,
                                layout=dict(shapes=[highlight_shape(k)])))

fig_tbl.frames = frames_tbl
fig_tbl.update_layout(
    title="head_dim=8 (4 pairs), 4 real token positions -- every scalar, and the pairs they form",
    shapes=[highlight_shape(0)],
    width=1150, height=520,
    updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=1.15, buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(steps=[
        dict(method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True))],
             label=row_labels[k])
        for k, f in enumerate(frames_tbl)
    ], currentvalue=dict(prefix="showing: "))],
)
fig_tbl.show()

## Extra: how does this change the attention score?

Once RoPE rotates $q$ (and $k$), the next real step is just a dot product:
$\text{score}(i,j) = q_i \cdot k_j / \sqrt{\text{head\_dim}}$. Reusing the
exact same 4-token, `head_dim=8` vectors from the table above as *both*
$q$ and $k$ (a simplification -- a real model's $q$/$k$ come from two
separate learned projections, but reusing one vector cleanly isolates what
rotation *alone* changes) turns the rotated numbers you just saw into a
real 4x4 attention-score table.

Left: the score **before** RoPE -- every position's un-rotated content is
identical $(1,1,\dots,1)$, so every cell is the exact same number; position
tells the dot product nothing at all. Right: the score **after** RoPE --
now the value genuinely depends on *how far apart* the two positions are
(exactly Graph 5's relative-distance property, now shown for this same
worked example). Press **Play** (or drag the slider) and watch both tables
fill in one query-position row at a time, so you can see each new row's
numbers appear as it's "computed."

In [219]:
scale_tbl = math.sqrt(head_dim_tbl)
attn_before_full = (q_tbl @ q_tbl.T).numpy() / scale_tbl   # every row/col identical -- content never changes with position
attn_after_full = (q_tbl_rot @ q_tbl_rot.T) / scale_tbl    # q_tbl_rot from the cell above: real per-position rotated vectors

print(f"before RoPE: every cell is identical -- (1,...,1) . (1,...,1) / sqrt(8) = {attn_before_full[0, 0]:.3f}\n")

print("after RoPE, worked by hand for one cell (query=pos 1, key=pos 2):")
terms = " + ".join(f"({q_tbl_rot[1, d]:.3f}*{q_tbl_rot[2, d]:.3f})" for d in range(head_dim_tbl))
manual_cell = sum(q_tbl_rot[1, d] * q_tbl_rot[2, d] for d in range(head_dim_tbl)) / scale_tbl
print(f"  [{terms}] / sqrt(8)\n  = {manual_cell:.3f}  (matches attn_after_full[1, 2] = {attn_after_full[1, 2]:.3f})")
assert abs(manual_cell - attn_after_full[1, 2]) < 1e-4

print("\nfull after-RoPE attention table:")
for i in range(seq_len_tbl):
    print(f"  {row_labels[i]}: [" + ", ".join(f"{v:+.3f}" for v in attn_after_full[i]) + "]")

print("\nrelative-position check -- every diagonal (constant key_pos - query_pos) is identical here,")
print("because q and k share the exact same identity-like content ((1,...,1)) at every position:")
for offset in range(-(seq_len_tbl - 1), seq_len_tbl):
    vals = [attn_after_full[i, i + offset] for i in range(seq_len_tbl) if 0 <= i + offset < seq_len_tbl]
    assert max(vals) - min(vals) < 1e-4, f"offset {offset} varies: {vals}"
    print(f"  key_pos - query_pos = {offset:+d}: " + ", ".join(f"{v:.3f}" for v in vals) + "  (all equal)")

before RoPE: every cell is identical -- (1,...,1) . (1,...,1) / sqrt(8) = 2.828

after RoPE, worked by hand for one cell (query=pos 1, key=pos 2):
  [(-0.301*-1.325) + (0.895*0.781) + (0.990*0.980) + (0.999*0.998) + (1.382*0.493) + (1.095*1.179) + (1.010*1.020) + (1.001*1.002)] / sqrt(8)
  = 2.500  (matches attn_after_full[1, 2] = 2.500)

full after-RoPE attention table:
  pos 0: [+2.828, +2.500, +1.813, +1.389]
  pos 1: [+2.500, +2.828, +2.500, +1.813]
  pos 2: [+1.813, +2.500, +2.828, +2.500]
  pos 3: [+1.389, +1.813, +2.500, +2.828]

relative-position check -- every diagonal (constant key_pos - query_pos) is identical here,
because q and k share the exact same identity-like content ((1,...,1)) at every position:
  key_pos - query_pos = -3: 1.389  (all equal)
  key_pos - query_pos = -2: 1.813, 1.813  (all equal)
  key_pos - query_pos = -1: 2.500, 2.500, 2.500  (all equal)
  key_pos - query_pos = +0: 2.828, 2.828, 2.828, 2.828  (all equal)
  key_pos - query_pos = +1: 2.500, 2.500, 2.5

In [220]:
def masked_rows(mat, k):
    m = np.array(mat, dtype=float).copy()
    m[k + 1:, :] = np.nan
    return m


def masked_text(text_mat, k):
    return [row if i <= k else ["" for _ in row] for i, row in enumerate(text_mat)]


text_before = [[f"{v:.3f}" for v in row] for row in attn_before_full]
text_after = [[f"{v:.3f}" for v in row] for row in attn_after_full]
shared_zmin, shared_zmax = attn_after_full.min(), attn_after_full.max()

fig_attn = make_subplots(rows=1, cols=2, subplot_titles=[
    "attention score BEFORE RoPE (position-blind)", "attention score AFTER RoPE (position-aware)",
])
fig_attn.add_trace(go.Heatmap(
    z=masked_rows(attn_before_full, 0), x=row_labels, y=row_labels,
    text=masked_text(text_before, 0), texttemplate="%{text}",
    colorscale="RdBu", zmid=0, zmin=shared_zmin, zmax=shared_zmax, showscale=False,
), row=1, col=1)
fig_attn.add_trace(go.Heatmap(
    z=masked_rows(attn_after_full, 0), x=row_labels, y=row_labels,
    text=masked_text(text_after, 0), texttemplate="%{text}",
    colorscale="RdBu", zmid=0, zmin=shared_zmin, zmax=shared_zmax,
    colorbar=dict(title="q.k/sqrt(head_dim)"),
), row=1, col=2)
fig_attn.update_xaxes(title="key position", row=1, col=1)
fig_attn.update_xaxes(title="key position", row=1, col=2)
fig_attn.update_yaxes(title="query position", autorange="reversed", row=1, col=1)
fig_attn.update_yaxes(title="query position", autorange="reversed", row=1, col=2)

frames_attn = [
    go.Frame(
        name=str(k),
        data=[
            go.Heatmap(z=masked_rows(attn_before_full, k), text=masked_text(text_before, k)),
            go.Heatmap(z=masked_rows(attn_after_full, k), text=masked_text(text_after, k)),
        ],
        traces=[0, 1],
    )
    for k in range(seq_len_tbl)
]

fig_attn.frames = frames_attn
fig_attn.update_layout(
    title="how RoPE changes the attention score (identity-like content: every pair starts at (1,1))",
    width=1000, height=460,
    updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=1.15, buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(steps=[
        dict(method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))],
             label=row_labels[k])
        for k in range(seq_len_tbl)
    ], currentvalue=dict(prefix="rows filled through: "))],
)
fig_attn.show()

### Now with a real matrix, not just the identity-like case

The table above used the same all-ones vector for every token, so the
"before" table came out completely uniform and the "after" table showed a
suspiciously clean pattern (every diagonal identical) -- a special case
that only happens because $q$ and $k$ share the exact same content at
every position. Below, $q$ and $k$ are instead two genuinely different,
real random vectors per token, so nothing is shared or trivial. The
"before" table is now a real, varied matrix, and the "after" table no
longer has that clean diagonal pattern -- but the *actual* RoPE guarantee
still holds: keep these same real vectors and shift all 4 positions by
the same amount (same relative spacing), and the score matrix comes out
*exactly* unchanged, checked explicitly below.

In [221]:
torch.manual_seed(7)
q_real = torch.randn(seq_len_tbl, head_dim_tbl)   # a genuinely different, real vector per token -- not all-ones
k_real = torch.randn(seq_len_tbl, head_dim_tbl)   # a separate real vector too, same as a real model's q != k
q_real_rot, k_real_rot = rope_tbl.apply_to_qk(q_real, k_real, positions_tbl)
q_real_rot, k_real_rot = q_real_rot.numpy(), k_real_rot.numpy()

attn_before_real = (q_real.numpy() @ k_real.numpy().T) / scale_tbl   # no rotation at all -- content only
attn_after_real = (q_real_rot @ k_real_rot.T) / scale_tbl             # real per-position rotated q/k

print("before RoPE (raw, un-rotated -- q and k are two different real random vectors per token):")
for i in range(seq_len_tbl):
    print(f"  {row_labels[i]}: [" + ", ".join(f"{v:+.3f}" for v in attn_before_real[i]) + "]")

print("\nafter RoPE, worked by hand for one cell (query=pos 1, key=pos 2):")
terms = " + ".join(f"({q_real_rot[1, d]:.3f}*{k_real_rot[2, d]:.3f})" for d in range(head_dim_tbl))
manual_cell_real = sum(q_real_rot[1, d] * k_real_rot[2, d] for d in range(head_dim_tbl)) / scale_tbl
print(f"  [{terms}] / sqrt(8)\n  = {manual_cell_real:.3f}  (matches attn_after_real[1, 2] = {attn_after_real[1, 2]:.3f})")
assert abs(manual_cell_real - attn_after_real[1, 2]) < 1e-4

print("\nfull after-RoPE attention table (q != k, a real, non-uniform, non-symmetric matrix):")
for i in range(seq_len_tbl):
    print(f"  {row_labels[i]}: [" + ", ".join(f"{v:+.3f}" for v in attn_after_real[i]) + "]")

# translation-invariance check (Graph 5's real property, reproduced for this exact real q/k pair):
# the SAME 4 real content vectors, shifted to later positions but with the same spacing, must
# give the identical score matrix -- NOT a diagonal-constant/Toeplitz pattern, which only happened
# in the identity-like example above because every token shared identical content.
shift = 5
q_shifted_rot, k_shifted_rot = rope_tbl.apply_to_qk(q_real, k_real, positions_tbl + shift)
attn_after_real_shifted = (q_shifted_rot.numpy() @ k_shifted_rot.numpy().T) / scale_tbl
max_diff_real = np.abs(attn_after_real - attn_after_real_shifted).max()
print(f"\ntranslation-invariance check: same 4 real vectors, positions shifted by +{shift}")
print(f"max|attn(pos 0..3) - attn(pos {shift}..{shift + 3})| = {max_diff_real:.2e}  (same spacing -> same scores)")
assert max_diff_real < 1e-4

before RoPE (raw, un-rotated -- q and k are two different real random vectors per token):
  pos 0: [-0.796, +1.422, +0.042, +0.161]
  pos 1: [-0.766, -1.220, -0.729, +1.466]
  pos 2: [-0.365, +1.906, -0.690, -1.310]
  pos 3: [-0.561, -1.148, +1.694, -0.704]

after RoPE, worked by hand for one cell (query=pos 1, key=pos 2):
  [(-0.096*2.771) + (-0.634*-0.044) + (-1.162*-0.008) + (-0.461*-0.386) + (1.162*1.072) + (0.954*-1.091) + (0.219*-0.309) + (1.090*0.934)] / sqrt(8)
  = 0.391  (matches attn_after_real[1, 2] = 0.391)

full after-RoPE attention table (q != k, a real, non-uniform, non-symmetric matrix):
  pos 0: [-0.796, +1.245, -1.021, +1.293]
  pos 1: [-0.566, -1.220, +0.391, +0.319]
  pos 2: [-0.999, +1.725, -0.690, -0.297]
  pos 3: [+0.842, -2.522, +0.786, -0.704]

translation-invariance check: same 4 real vectors, positions shifted by +5
max|attn(pos 0..3) - attn(pos 5..8)| = 2.68e-07  (same spacing -> same scores)


In [222]:
text_before_real = [[f"{v:.3f}" for v in row] for row in attn_before_real]
text_after_real = [[f"{v:.3f}" for v in row] for row in attn_after_real]
shared_zmin_real, shared_zmax_real = attn_after_real.min(), attn_after_real.max()

fig_attn_real = make_subplots(rows=1, cols=2, subplot_titles=[
    "attention score BEFORE RoPE (position-blind)", "attention score AFTER RoPE (position-aware)",
])
fig_attn_real.add_trace(go.Heatmap(
    z=masked_rows(attn_before_real, 0), x=row_labels, y=row_labels,
    text=masked_text(text_before_real, 0), texttemplate="%{text}",
    colorscale="RdBu", zmid=0, zmin=shared_zmin_real, zmax=shared_zmax_real, showscale=False,
), row=1, col=1)
fig_attn_real.add_trace(go.Heatmap(
    z=masked_rows(attn_after_real, 0), x=row_labels, y=row_labels,
    text=masked_text(text_after_real, 0), texttemplate="%{text}",
    colorscale="RdBu", zmid=0, zmin=shared_zmin_real, zmax=shared_zmax_real,
    colorbar=dict(title="q.k/sqrt(head_dim)"),
), row=1, col=2)
fig_attn_real.update_xaxes(title="key position", row=1, col=1)
fig_attn_real.update_xaxes(title="key position", row=1, col=2)
fig_attn_real.update_yaxes(title="query position", autorange="reversed", row=1, col=1)
fig_attn_real.update_yaxes(title="query position", autorange="reversed", row=1, col=2)

frames_attn_real = [
    go.Frame(
        name=str(k),
        data=[
            go.Heatmap(z=masked_rows(attn_before_real, k), text=masked_text(text_before_real, k)),
            go.Heatmap(z=masked_rows(attn_after_real, k), text=masked_text(text_after_real, k)),
        ],
        traces=[0, 1],
    )
    for k in range(seq_len_tbl)
]

fig_attn_real.frames = frames_attn_real
fig_attn_real.update_layout(
    title="how RoPE changes the attention score (a real matrix -- q and k genuinely different per token)",
    width=1000, height=460,
    updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=1.15, buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(steps=[
        dict(method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))],
             label=row_labels[k])
        for k in range(seq_len_tbl)
    ], currentvalue=dict(prefix="rows filled through: "))],
)
fig_attn_real.show()

## Extra: does attention actually decay with distance?

A common claim about RoPE is that attention scores decay as two tokens
get farther apart. Worth checking rather than assuming: does this hold
for *arbitrary* content?

**It does not, for random content.** Rotation preserves lengths and
angles (Graph 4), and a standard Gaussian random vector's distribution is
perfectly rotationally symmetric -- so rotating $k$ by *any* angle before
the dot product doesn't change the statistical distribution of $q \cdot k$
at all. Averaging $|\text{score}|$ over 300 independent random $(q,k)$
pairs (thin gray lines: 15 individual pairs, each just oscillating
forever, never shrinking) confirms this directly.

**The real decay claim is about a different kind of pair: one where every
frequency happens to start perfectly aligned.** Reusing the exact
$q=k=(1,\dots,1)$ vector from the earlier identity-content table (every
one of `head_dim=16`'s 8 frequency pairs contributes its full positive
value at distance 0) *does* show a real, shrinking envelope: the
different-frequency pairs start perfectly in phase, then spread apart at
different rates and increasingly cancel each other out. This is the
actual mechanism behind RoPE's "long-term decay" property from the
original paper -- not that attention *always* weakens with distance, but
that the *most constructively-aligned* score's upper bound shrinks, a
purely geometric consequence of combining several rotation frequencies,
with no learned parameters involved.

In [223]:
torch.manual_seed(123)
n_samples = 5000  # bigger sample -> smoother, more statistically robust random-content baseline
max_distance = 22000  # past period_slowest (~19869) established in Graph 3 -- see if it ever recovers
scale_decay = math.sqrt(head_dim)
distances = list(range(0, max_distance + 1, 5))

# Random, independent q/k content -- rotation preserves an isotropic Gaussian's distribution
# exactly, so this should NOT decay on average. Worth confirming directly rather than assuming.
q_samples = torch.randn(n_samples, 1, head_dim)
k_samples = torch.randn(n_samples, 1, head_dim)
scores_per_sample = torch.zeros(n_samples, len(distances))
for i, d in enumerate(distances):
    k_rot_d, _ = rope_real.apply_to_qk(k_samples, k_samples, torch.tensor([float(d)]))
    scores_per_sample[:, i] = (q_samples.squeeze(1) * k_rot_d.squeeze(1)).sum(dim=-1) / scale_decay
mean_abs_score = scores_per_sample.abs().mean(dim=0).numpy()

# A FIXED, deliberately-aligned q=k=(1,...,1) pair (same convention as the earlier identity
# table) -- every one of its 8 frequency pairs starts perfectly constructively aligned at
# distance 0, so THIS is where the real "long-term decay" envelope should show up.
q_fixed = torch.ones(1, head_dim)
scores_fixed = []
for d in distances:
    k_rot_fixed, _ = rope_real.apply_to_qk(q_fixed, q_fixed, torch.tensor([float(d)]))
    scores_fixed.append(((q_fixed * k_rot_fixed).sum() / scale_decay).item())
scores_fixed = np.array(scores_fixed)

print(f"random content:  mean |score| at distance 0 = {mean_abs_score[0]:.3f}, "
      f"at distance {max_distance} = {mean_abs_score[-1]:.3f}  (flat -- no real decay)")
print(f"aligned pair:     score at distance 0 = {scores_fixed[0]:.3f}, "
      f"peak |score| in distances 0-100 = {np.abs(scores_fixed[:21]).max():.3f}, "
      f"peak |score| in distances 1000-1100 = {np.abs(scores_fixed[200:221]).max():.3f}")
idx_near_slowest = int(period_slowest // 5)
print(f"peak |score| near the slowest pair's own period (~{period_slowest:.0f}): "
      f"{np.abs(scores_fixed[max(0, idx_near_slowest - 20):idx_near_slowest + 20]).max():.3f}  "
      f"(does it recover once the LAST pair completes a full turn?)")

fig_decay = go.Figure()
rng = np.random.default_rng(0)
sample_idx = rng.choice(n_samples, size=15, replace=False)
for idx in sample_idx:
    fig_decay.add_trace(go.Scatter(x=distances, y=scores_per_sample[idx].numpy(), mode="lines",
                                    line=dict(color="lightgray", width=1), opacity=0.5,
                                    showlegend=False, hoverinfo="skip"))
fig_decay.add_trace(go.Scatter(x=distances, y=mean_abs_score, mode="lines",
                                line=dict(color="royalblue", width=2, dash="dash"),
                                name=f"mean |score|, {n_samples} random q/k pairs (rotation preserves isotropy -- stays flat)"))
fig_decay.add_trace(go.Scatter(x=distances, y=scores_fixed, mode="lines",
                                line=dict(color="firebrick", width=2),
                                name="score for ONE fixed, deliberately-aligned q=k=(1,...,1) pair"))
fig_decay.add_vline(x=SMOKE_TEST.max_seq_len, line=dict(color="gray", dash="dot"),
                     annotation_text="SMOKE_TEST max_seq_len=128")
fig_decay.update_layout(
    title="Random content doesn't decay with distance -- a deliberately-aligned pair does",
    xaxis_title="relative distance (key position - query position)",
    yaxis_title="q . k / sqrt(head_dim)",
    width=950, height=480,
)
fig_decay.show()

random content:  mean |score| at distance 0 = 0.792, at distance 22000 = 0.782  (flat -- no real decay)
aligned pair:     score at distance 0 = 4.000, peak |score| in distances 0-100 = 4.000, peak |score| in distances 1000-1100 = 1.243
peak |score| near the slowest pair's own period (~19869): 2.268  (does it recover once the LAST pair completes a full turn?)


### Unrolling the rotation into a wave

Every earlier picture drew a rotating arrow going around a circle. The
exact same rotation looks different -- and makes "phase cancellation"
much easier to see -- if you stop drawing the circle and instead plot
just $\cos(\text{position} \times \text{freq}_i)$ as an ordinary wave,
with position along the x-axis, like a time-lapse instead of a clock face.

**Top panel:** each of `head_dim=16`'s 8 frequency pairs, position 0-150.
Pair 0 (fastest) completes a full cycle every ~6.3 positions -- 24 cycles
in this window. Pair 7 (slowest) has barely moved off its starting value.
**At position 0, every single wave starts at its peak** ($\cos(0)=1$) --
every frequency is perfectly aligned, which is exactly why the
aligned-pair example started at its maximum possible score. The bold
black line is these 8 waves added together (scaled by 0.5, matching the
real formula: `score(d) = 0.5 * sum_i cos(d * freq_i)`) -- the same
quantity as `scores_fixed` from the previous cell.

**Bottom panel:** that identical sum, continued out to distance 3000
(literally the same numbers as the previous cell's red curve). Early on
(shaded region = the window shown above), the waves haven't spread out
enough to disagree much, so the sum stays near its maximum. Farther out,
the fast pairs have already cycled through many peaks and troughs while
the slow pairs have barely moved -- so at any given position some pairs
are near +1, others near -1, and the sum increasingly cancels itself out.
That's "phase cancellation": not any single wave decaying, but several
different-speed waves drifting out of sync with each other.

In [224]:
positions_wave = np.arange(0, 151)
freqs = rope_real.inv_freq.numpy()  # same 8 frequencies as scores_fixed, fastest to slowest
wave_colors = pc.sample_colorscale("Viridis", [i / (len(freqs) - 1) for i in range(len(freqs))])

individual_waves = np.cos(positions_wave[:, None] * freqs[None, :])  # (151, 8)
combined_wave = 0.5 * individual_waves.sum(axis=1)  # exact same formula behind scores_fixed

# Rolling max |score| envelope at full position resolution (not the sparse step-5 `distances`
# grid), so it can't alias against the fastest pair's ~6.3-position period.
dense_positions = np.arange(0, max_distance + 1)
dense_wave = 0.5 * np.cos(dense_positions[:, None] * freqs[None, :]).sum(axis=1)
window = 65
padded_abs = np.pad(np.abs(dense_wave), (window // 2, window // 2), mode="edge")
envelope = np.lib.stride_tricks.sliding_window_view(padded_abs, window).max(axis=1)

fig_waves = make_subplots(
    rows=2, cols=1, row_heights=[0.55, 0.45], vertical_spacing=0.12,
    subplot_titles=[
        "Each frequency pair as a wave (position 0-150) -- amplitude over position, not a circle",
        "The raw sum oscillates (gray) -- the envelope of its nearby peaks (orange) is what decays",
    ],
)

for i in range(len(freqs)):
    fig_waves.add_trace(go.Scatter(x=positions_wave, y=individual_waves[:, i], mode="lines",
                                    line=dict(color=wave_colors[i], width=1.5),
                                    name=f"pair {i} (freq={freqs[i]:.4f})"), row=1, col=1)
fig_waves.add_trace(go.Scatter(x=positions_wave, y=combined_wave, mode="lines",
                                line=dict(color="black", width=3),
                                name="sum of all 8 (x0.5) = scores_fixed"), row=1, col=1)

fig_waves.add_trace(go.Scatter(x=distances, y=scores_fixed, mode="lines",
                                line=dict(color="gray", width=1), opacity=0.7,
                                name="actual score (oscillates -- does NOT shrink point-by-point)"), row=2, col=1)
fig_waves.add_trace(go.Scatter(x=dense_positions, y=envelope, mode="lines",
                                line=dict(color="orange", width=2, dash="dash"),
                                name="envelope: best |score| reachable nearby (THIS is what decays)"), row=2, col=1)
fig_waves.add_trace(go.Scatter(x=dense_positions, y=-envelope, mode="lines",
                                line=dict(color="orange", width=2, dash="dash"), showlegend=False), row=2, col=1)
fig_waves.add_vrect(x0=0, x1=150, fillcolor="lightgray", opacity=0.3, line_width=0, row=2, col=1,
                     annotation_text="range shown above", annotation_position="top left")

fig_waves.update_xaxes(title="position m", row=1, col=1)
fig_waves.update_yaxes(title="cos(m x freq_i)", range=[-1.2, 1.2], row=1, col=1)
fig_waves.update_xaxes(title="position (relative distance)", row=2, col=1)
fig_waves.update_yaxes(title="sum = score for the aligned pair", row=2, col=1)

fig_waves.update_layout(
    title="Unrolling the rotation into waves: aligned at m=0, drifting apart, increasingly canceling",
    width=980, height=820,
)
fig_waves.show()

print(f"sum at position 0: {combined_wave[0]:.3f}  (matches scores_fixed[0] = {scores_fixed[0]:.3f})")
print(f"envelope (best |score| reachable nearby) at distance 0:     {envelope[0]:.3f}")
print(f"envelope (best |score| reachable nearby) at distance 1000:  {envelope[1000]:.3f}")
print(f"envelope (best |score| reachable nearby) at distance 3000:  {envelope[3000]:.3f}")
print(f"envelope (best |score| reachable nearby) at distance {idx_near_slowest * 5}: {envelope[idx_near_slowest * 5]:.3f}  "
      f"(partial recovery -- the slowest pair, period ~{period_slowest:.0f}, has come back near its start)")
print(f"envelope (best |score| reachable nearby) at distance {max_distance}: {envelope[max_distance]:.3f}")
print("the raw gray curve still swings up and down forever -- the shrinking (then partly recovering) ORANGE envelope is 'decay'")

sum at position 0: 4.000  (matches scores_fixed[0] = 4.000)
envelope (best |score| reachable nearby) at distance 0:     4.000
envelope (best |score| reachable nearby) at distance 1000:  1.581
envelope (best |score| reachable nearby) at distance 3000:  1.511
envelope (best |score| reachable nearby) at distance 19865: 2.621  (partial recovery -- the slowest pair, period ~19869, has come back near its start)
envelope (best |score| reachable nearby) at distance 22000: 2.005
the raw gray curve still swings up and down forever -- the shrinking (then partly recovering) ORANGE envelope is 'decay'


### Does this hold for REAL, trained content -- not synthetic vectors?

Everything above used made-up content: random Gaussian samples, or an
artificial all-ones vector deliberately constructed to align every
frequency. Worth checking against actual data: reusing the exact
pre-RoPE `layer0.q`/`layer0.k` tensors already pulled from the real
`runs/smoke_test/checkpoints/step_1000.pt` checkpoint in Part 3 (Graph 6),
plus that checkpoint's own `positional` module -- one real learned query
vector and one real learned key vector, swept across the same wide range
of simulated relative distances. No random numbers, no artificial
alignment -- just whatever these two real, trained vectors' geometry
actually produces.

In [225]:
q_real_tok = layer0.q[0, -1].clone()   # real learned query, head 0, last token
k_real_tok = layer0.k[0, 0].clone()    # real learned key, kv-head 0, first token
print(f"query token = '{tokens[-1]}', key token = '{tokens[0]}'  (from the real \"ROMEO:\" prompt)")
print(f"||q||={q_real_tok.norm().item():.3f}  ||k||={k_real_tok.norm().item():.3f}  "
      f"raw dot product with no RoPE at all = {(q_real_tok @ k_real_tok).item() / math.sqrt(real_head_dim):.3f}")

scores_real_tok = []
for d in distances:
    k_rot_real, _ = positional.apply_to_qk(k_real_tok.unsqueeze(0), k_real_tok.unsqueeze(0), torch.tensor([float(d)]))
    scores_real_tok.append(((q_real_tok * k_rot_real.squeeze(0)).sum() / math.sqrt(real_head_dim)).item())
scores_real_tok = np.array(scores_real_tok)

print(f"\nreal-content score at distance 0:     {scores_real_tok[0]:.3f}")
print(f"real-content score at distance 1000:  {scores_real_tok[200]:.3f}")
print(f"real-content score at distance {idx_near_slowest * 5}: {scores_real_tok[idx_near_slowest]:.3f}  (near the slowest pair's own period)")
print(f"real-content score at distance {max_distance}: {scores_real_tok[-1]:.3f}")

fig_real_compare = go.Figure()
fig_real_compare.add_trace(go.Scatter(x=distances, y=mean_abs_score, mode="lines",
                                       line=dict(color="royalblue", width=2, dash="dash"),
                                       name=f"mean |score|, {n_samples} RANDOM q/k pairs (flat)"))
fig_real_compare.add_trace(go.Scatter(x=distances, y=scores_fixed, mode="lines",
                                       line=dict(color="firebrick", width=1.5), opacity=0.6,
                                       name="score, SYNTHETIC all-ones aligned pair"))
fig_real_compare.add_trace(go.Scatter(x=distances, y=scores_real_tok, mode="lines",
                                       line=dict(color="black", width=2),
                                       name=f"score, REAL learned q='{tokens[-1]}' / k='{tokens[0]}' (checkpoint step_1000)"))
fig_real_compare.add_vline(x=SMOKE_TEST.max_seq_len, line=dict(color="gray", dash="dot"),
                            annotation_text="SMOKE_TEST max_seq_len=128")
fig_real_compare.update_layout(
    title="Real trained Q/K content vs. the two synthetic cases",
    xaxis_title="relative distance (key position - query position)",
    yaxis_title="q . k / sqrt(head_dim)",
    width=980, height=500,
)
fig_real_compare.show()

query token = ':', key token = 'R'  (from the real "ROMEO:" prompt)
||q||=3.264  ||k||=5.051  raw dot product with no RoPE at all = 3.090

real-content score at distance 0:     3.090
real-content score at distance 1000:  0.341
real-content score at distance 19865: 0.449  (near the slowest pair's own period)
real-content score at distance 22000: 0.750


### Zooming into the first 3000 positions

The comparison above spans the full 22,000-position range, which
compresses the early, most informative part of the curve into a thin
sliver on the left edge. Same three curves (random, synthetic aligned,
real learned content), same underlying numbers -- just the x-axis
clipped to 0-3000 so the actual shape of the early decay is readable.

In [226]:
zoom_limit = 3000
n_zoom = zoom_limit // 5 + 1  # distances steps by 5

fig_real_zoom = go.Figure()
fig_real_zoom.add_trace(go.Scatter(x=distances[:n_zoom], y=mean_abs_score[:n_zoom], mode="lines",
                                    line=dict(color="royalblue", width=2, dash="dash"),
                                    name=f"mean |score|, {n_samples} RANDOM q/k pairs (flat)"))
fig_real_zoom.add_trace(go.Scatter(x=distances[:n_zoom], y=scores_fixed[:n_zoom], mode="lines",
                                    line=dict(color="firebrick", width=1.5), opacity=0.6,
                                    name="score, SYNTHETIC all-ones aligned pair"))
fig_real_zoom.add_trace(go.Scatter(x=distances[:n_zoom], y=scores_real_tok[:n_zoom], mode="lines",
                                    line=dict(color="black", width=2),
                                    name=f"score, REAL learned q='{tokens[-1]}' / k='{tokens[0]}' (checkpoint step_1000)"))
fig_real_zoom.add_vline(x=SMOKE_TEST.max_seq_len, line=dict(color="gray", dash="dot"),
                         annotation_text="SMOKE_TEST max_seq_len=128")
fig_real_zoom.update_layout(
    title=f"Same three curves, zoomed to the first {zoom_limit} positions",
    xaxis_title="relative distance (key position - query position)",
    yaxis_title="q . k / sqrt(head_dim)",
    width=980, height=500,
)
fig_real_zoom.show()

print(f"real-content score at distance 0:   {scores_real_tok[0]:.3f}")
print(f"real-content score at distance 128 (max_seq_len): {scores_real_tok[128 // 5]:.3f}")
print(f"real-content score at distance {zoom_limit}: {scores_real_tok[n_zoom - 1]:.3f}")

real-content score at distance 0:   3.090
real-content score at distance 128 (max_seq_len): 1.326
real-content score at distance 3000: 0.391


### Smoothing out the oscillation: the overall flow

Both the synthetic and real curves above oscillate rapidly, which makes
the underlying trend hard to read at a glance. Taking a rolling **mean**
of $|\text{score}|$ (not the earlier rolling *max* envelope -- this one
averages out the wiggle instead of tracing its peaks) for both the
synthetic aligned pair and the real learned pair, over the full
0-22,000 range, makes the overall shape of each directly comparable in
one glance: same starting point, same general downward trend, but very
different steepness and noise floor.

In [227]:
window_samples = 13  # ~65 raw positions (13 samples x step-5), same smoothing scale as the earlier envelope
kernel = np.ones(window_samples) / window_samples
pad = window_samples // 2

# edge-pad (not zero-pad) before convolving, so the average near distance 0 isn't
# artificially pulled down by implicit zeros -- same fix already used for `envelope` above.
avg_fixed = np.convolve(np.pad(np.abs(scores_fixed), (pad, pad), mode="edge"), kernel, mode="valid")
avg_real = np.convolve(np.pad(np.abs(scores_real_tok), (pad, pad), mode="edge"), kernel, mode="valid")

fig_avg = go.Figure()
fig_avg.add_trace(go.Scatter(x=distances, y=np.abs(scores_fixed), mode="lines",
                              line=dict(color="firebrick", width=1), opacity=0.25, showlegend=False, hoverinfo="skip"))
fig_avg.add_trace(go.Scatter(x=distances, y=np.abs(scores_real_tok), mode="lines",
                              line=dict(color="black", width=1), opacity=0.25, showlegend=False, hoverinfo="skip"))
fig_avg.add_trace(go.Scatter(x=distances, y=[mean_abs_score.mean()] * len(distances), mode="lines",
                              line=dict(color="royalblue", width=2, dash="dash"),
                              name=f"mean |score|, {n_samples} random q/k pairs (flat, for reference)"))
fig_avg.add_trace(go.Scatter(x=distances, y=avg_fixed, mode="lines",
                              line=dict(color="firebrick", width=3),
                              name="rolling avg |score|, SYNTHETIC all-ones aligned pair"))
fig_avg.add_trace(go.Scatter(x=distances, y=avg_real, mode="lines",
                              line=dict(color="black", width=3),
                              name=f"rolling avg |score|, REAL learned q='{tokens[-1]}' / k='{tokens[0]}'"))
fig_avg.add_vline(x=SMOKE_TEST.max_seq_len, line=dict(color="gray", dash="dot"),
                   annotation_text="SMOKE_TEST max_seq_len=128")
fig_avg.update_layout(
    title="Rolling average |score| -- the overall flow, oscillation smoothed out",
    xaxis_title="relative distance (key position - query position)",
    yaxis_title="rolling mean of |q . k / sqrt(head_dim)|",
    width=980, height=500,
)
fig_avg.show()

print(f"synthetic aligned pair: rolling avg at distance 0 = {avg_fixed[0]:.3f}, "
      f"at distance 3000 = {avg_fixed[600]:.3f}, at distance {max_distance} = {avg_fixed[-1]:.3f}")
print(f"real learned pair:      rolling avg at distance 0 = {avg_real[0]:.3f}, "
      f"at distance 3000 = {avg_real[600]:.3f}, at distance {max_distance} = {avg_real[-1]:.3f}")

synthetic aligned pair: rolling avg at distance 0 = 3.203, at distance 3000 = 0.511, at distance 22000 = 0.754
real learned pair:      rolling avg at distance 0 = 2.439, at distance 3000 = 0.869, at distance 22000 = 0.689


### Pushing it out to 100,000 positions

One more: the same rolling-average "overall flow" comparison, extended to
100,000 positions -- far enough to fit roughly **5 full revolutions** of
the slowest frequency pair (period ~19,869, from Graph 3). Vertical green
lines mark each multiple of that period. If the earlier "partial recovery"
near one period was real and not a fluke, it should show up again --
fainter or stronger -- near each subsequent multiple too.

In [228]:
torch.manual_seed(123)
max_distance_100k = 100_000
distances_100k = list(range(0, max_distance_100k + 1, 5))

q_samples_100k = torch.randn(n_samples, 1, head_dim)
k_samples_100k = torch.randn(n_samples, 1, head_dim)
scores_per_sample_100k = torch.zeros(n_samples, len(distances_100k))
scores_fixed_100k = []
scores_real_tok_100k = []
for i, d in enumerate(distances_100k):
    d_t = torch.tensor([float(d)])
    k_rot_d, _ = rope_real.apply_to_qk(k_samples_100k, k_samples_100k, d_t)
    scores_per_sample_100k[:, i] = (q_samples_100k.squeeze(1) * k_rot_d.squeeze(1)).sum(dim=-1) / scale_decay
    k_rot_fixed, _ = rope_real.apply_to_qk(q_fixed, q_fixed, d_t)
    scores_fixed_100k.append(((q_fixed * k_rot_fixed).sum() / scale_decay).item())
    k_rot_real, _ = positional.apply_to_qk(k_real_tok.unsqueeze(0), k_real_tok.unsqueeze(0), d_t)
    scores_real_tok_100k.append(((q_real_tok * k_rot_real.squeeze(0)).sum() / math.sqrt(real_head_dim)).item())
mean_abs_score_100k = scores_per_sample_100k.abs().mean(dim=0).numpy()
scores_fixed_100k = np.array(scores_fixed_100k)
scores_real_tok_100k = np.array(scores_real_tok_100k)

avg_fixed_100k = np.convolve(np.pad(np.abs(scores_fixed_100k), (pad, pad), mode="edge"), kernel, mode="valid")
avg_real_100k = np.convolve(np.pad(np.abs(scores_real_tok_100k), (pad, pad), mode="edge"), kernel, mode="valid")

period_multiples = [n * period_slowest for n in range(1, 6) if n * period_slowest <= max_distance_100k]

fig_avg_100k = go.Figure()
fig_avg_100k.add_trace(go.Scatter(x=distances_100k, y=np.abs(scores_fixed_100k), mode="lines",
                                   line=dict(color="firebrick", width=1), opacity=0.2, showlegend=False, hoverinfo="skip"))
fig_avg_100k.add_trace(go.Scatter(x=distances_100k, y=np.abs(scores_real_tok_100k), mode="lines",
                                   line=dict(color="black", width=1), opacity=0.2, showlegend=False, hoverinfo="skip"))
fig_avg_100k.add_trace(go.Scatter(x=distances_100k, y=[mean_abs_score_100k.mean()] * len(distances_100k), mode="lines",
                                   line=dict(color="royalblue", width=2, dash="dash"),
                                   name=f"mean |score|, {n_samples} random q/k pairs (flat, for reference)"))
fig_avg_100k.add_trace(go.Scatter(x=distances_100k, y=avg_fixed_100k, mode="lines",
                                   line=dict(color="firebrick", width=3),
                                   name="rolling avg |score|, SYNTHETIC all-ones aligned pair"))
fig_avg_100k.add_trace(go.Scatter(x=distances_100k, y=avg_real_100k, mode="lines",
                                   line=dict(color="black", width=3),
                                   name=f"rolling avg |score|, REAL learned q='{tokens[-1]}' / k='{tokens[0]}'"))
for n, x in enumerate(period_multiples, start=1):
    fig_avg_100k.add_vline(x=x, line=dict(color="seagreen", dash="dot", width=1),
                            annotation_text=f"{n}x period", annotation_position="top")
fig_avg_100k.add_vline(x=SMOKE_TEST.max_seq_len, line=dict(color="gray", dash="dot"),
                        annotation_text="max_seq_len=128", annotation_position="bottom")
fig_avg_100k.update_layout(
    title="Rolling average |score| out to 100,000 positions -- ~5 full periods of the slowest pair",
    xaxis_title="relative distance (key position - query position)",
    yaxis_title="rolling mean of |q . k / sqrt(head_dim)|",
    width=1050, height=520,
)
fig_avg_100k.show()

print(f"slowest pair's period: {period_slowest:.0f}  ->  multiples within range: "
      + ", ".join(f"{x:.0f}" for x in period_multiples))
print(f"\nsynthetic aligned pair rolling avg near each multiple:")
for n, x in enumerate(period_multiples, start=1):
    idx = min(range(len(distances_100k)), key=lambda i: abs(distances_100k[i] - x))
    print(f"  {n}x ({x:,.0f}): {avg_fixed_100k[idx]:.3f}")
print(f"\nreal learned pair rolling avg near each multiple:")
for n, x in enumerate(period_multiples, start=1):
    idx = min(range(len(distances_100k)), key=lambda i: abs(distances_100k[i] - x))
    print(f"  {n}x ({x:,.0f}): {avg_real_100k[idx]:.3f}")

slowest pair's period: 19869  ->  multiples within range: 19869, 39738, 59608, 79477, 99346

synthetic aligned pair rolling avg near each multiple:
  1x (19,869): 1.240
  2x (39,738): 1.199
  3x (59,608): 1.240
  4x (79,477): 0.704
  5x (99,346): 1.927

real learned pair rolling avg near each multiple:
  1x (19,869): 0.543
  2x (39,738): 0.968
  3x (59,608): 1.010
  4x (79,477): 0.818
  5x (99,346): 1.691


## Extra: does the REAL model actually break down past `max_seq_len`?

Every "decay"/"period" number above (`SMOKE_TEST.max_seq_len=128`,
`period_slowest~19,869`) was computed on synthetic distance sweeps. The
real, practical question those numbers exist to answer has never actually
been tested yet: **this exact trained checkpoint (`runs/smoke_test/
checkpoints/step_1000.pt`) never saw a sequence longer than 128 tokens
during training. What does it actually do if given one?**

`generate()` itself refuses to go past `model.config.max_seq_len` (its own
`Requirement 6.3` stop condition) -- but nothing about the *model* enforces
that: `RotaryEmbedding._angles` computes angles from `positions` on the fly
(no fixed-size table), and the causal mask (`qwenmoe/model/attention.py`)
is built fresh from the real `seq_len` every forward pass. So it's possible
to bypass `generate()` entirely and hand the real model a longer real
sequence directly. Using a real, verbatim ~300-character passage pulled
straight from this project's own training corpus (`.cache/data/
tiny_shakespeare.txt`, not fabricated) -- more than double `max_seq_len` --
lets us measure, not guess, what happens.

Graph 5 already proved attention scores depend only on *relative* distance,
never absolute position -- so the per-pair rotation math itself has no
reason to suddenly break at position 129. The real open question is
something Graph 5 never tested: **every query beyond position 128 now
competes against more keys in its softmax than this model ever trained
with.** Worth actually measuring rather than assuming either "it just
works" or "it falls apart."


In [229]:
corpus_path = Path(".cache/data/tiny_shakespeare.txt")
corpus_text = corpus_path.read_text()

# a real, verbatim passage from the SAME corpus this checkpoint trained on -- not fabricated
anchor = corpus_text.index("ROMEO:\nIs the day so young?")
long_prompt = corpus_text[anchor:anchor + 300]

print(f"model.config.max_seq_len = {model.config.max_seq_len}  (this checkpoint's real trained context length)")
print(f"real passage pulled from tiny_shakespeare.txt, {len(long_prompt)} characters:\n")
print(long_prompt)


model.config.max_seq_len = 128  (this checkpoint's real trained context length)
real passage pulled from tiny_shakespeare.txt, 300 characters:

ROMEO:
Is the day so young?

BENVOLIO:
But new struck nine.

ROMEO:
Ay me! sad hours seem long.
Was that my father that went hence so fast?

BENVOLIO:
It was. What sadness lengthens Romeo's hours?

ROMEO:
Not having that, which, having, makes them short.

BENVOLIO:
In love?

ROMEO:
Out--

BENVOLIO:



In [230]:
token_ids_long = tokenizer.encode(long_prompt)
print(f"encodes to {len(token_ids_long)} tokens ({len(token_ids_long) / model.config.max_seq_len:.2f}x max_seq_len)")

input_ids_long = torch.tensor([token_ids_long], dtype=torch.long)
with torch.no_grad():
    logits_long, _ = model(input_ids_long)  # one real forward pass, full ~300-token context -- bypasses generate()'s stop

targets_long = input_ids_long[0, 1:]
logits_for_targets = logits_long[0, :-1]
per_pos_loss = torch.nn.functional.cross_entropy(logits_for_targets, targets_long, reduction="none").detach().numpy()

within = per_pos_loss[: model.config.max_seq_len - 1]
beyond = per_pos_loss[model.config.max_seq_len - 1:]
print(f"\nmean real next-token loss, positions 1..{model.config.max_seq_len - 1} (WITHIN trained max_seq_len): {within.mean():.3f}")
print(f"mean real next-token loss, positions {model.config.max_seq_len}..{len(token_ids_long) - 1} (BEYOND trained max_seq_len): {beyond.mean():.3f}")
print(f"(random-init baseline would be ln(vocab_size={model.config.vocab_size}) = {math.log(model.config.vocab_size):.3f})")

fig_extrap_loss = go.Figure()
fig_extrap_loss.add_trace(go.Scatter(x=list(range(1, len(token_ids_long))), y=per_pos_loss, mode="lines+markers",
                                      line=dict(color="black", width=1), marker=dict(size=4),
                                      name="real per-position next-token loss"))
fig_extrap_loss.add_vline(x=model.config.max_seq_len, line=dict(color="firebrick", dash="dash"),
                           annotation_text=f"max_seq_len={model.config.max_seq_len} (never trained past here)")
fig_extrap_loss.update_layout(
    title=f'real per-position loss, "ROMEO:..." passage, positions 1..{len(token_ids_long) - 1}',
    xaxis_title="token position (predicting this position from everything before it)",
    yaxis_title="cross-entropy loss (nats)",
    width=980, height=440,
)
fig_extrap_loss.show()


encodes to 300 tokens (2.34x max_seq_len)

mean real next-token loss, positions 1..127 (WITHIN trained max_seq_len): 2.105
mean real next-token loss, positions 128..299 (BEYOND trained max_seq_len): 2.506
(random-init baseline would be ln(vocab_size=65) = 4.174)


In [231]:
with torch.no_grad():
    _, bundle_long = model.forward_with_trace(input_ids_long)

query_pos = len(token_ids_long) - 1  # the very last real token -- 171 positions past max_seq_len=128
attn_row = bundle_long.layers[0].attn_scores_post_softmax[0, query_pos, :query_pos + 1].detach().numpy()

fig_extrap_attn = go.Figure()
fig_extrap_attn.add_trace(go.Bar(x=list(range(query_pos + 1)), y=attn_row,
                                  marker=dict(color=attn_row, colorscale="Viridis"), showlegend=False))
fig_extrap_attn.add_vline(x=model.config.max_seq_len, line=dict(color="firebrick", dash="dash"),
                           annotation_text=f"max_seq_len={model.config.max_seq_len}")
fig_extrap_attn.update_layout(
    title=f"real attention from the LAST token (position {query_pos}) over its full {query_pos + 1}-token context "
          f"(head 0, layer 0)",
    xaxis_title="key position", yaxis_title="attention weight",
    width=980, height=380,
)
fig_extrap_attn.show()

top5 = np.argsort(attn_row)[-5:][::-1]
print(f"query position {query_pos} (real, {query_pos - model.config.max_seq_len} positions past max_seq_len)")
print("top-5 attended key positions:", [(int(p), round(float(attn_row[p]), 4)) for p in top5])
print(f"total attention mass on the most recent 20 tokens (positions {query_pos - 19}..{query_pos}): "
      f"{attn_row[-20:].sum():.4f}")
print(f"total attention mass on the first 20 tokens (positions 0..19): {attn_row[:20].sum():.4f}")


query position 299 (real, 171 positions past max_seq_len)
top-5 attended key positions: [(153, 0.0109), (299, 0.0091), (63, 0.0089), (59, 0.0088), (155, 0.0088)]
total attention mass on the most recent 20 tokens (positions 280..299): 0.0612
total attention mass on the first 20 tokens (positions 0..19): 0.0927


### What the real numbers actually show

Neither of the two "expected" stories is quite right:

- **Not "it just works" (perfect extrapolation).** Real mean next-token loss
  goes from **2.105** (positions 1-127, within `max_seq_len`) to **2.506**
  (positions 128-299, past it) -- a genuine, measurable ~19% jump the
  moment this model sees more context than it ever trained with.
- **Not "it falls apart" (catastrophic collapse) either.** 2.506 is still
  far below the random-init baseline of `ln(65) = 4.174` -- the model is
  still doing real, useful prediction 171 tokens past its trained length,
  just measurably worse than inside it.
- **The real attention row is surprisingly *diffuse*, not recency-spiked.**
  The last token's single largest weight is only 0.011 (over 300 keys,
  uniform would be ~0.0033) -- mildly peaked, not concentrated. More
  strikingly, this head puts *more* total mass on the first 20 tokens
  (0.093) than the most recent 20 (0.061) -- the opposite of the "recency
  bias" a well-behaved causal LM is often assumed to default to. That's a
  property of *this specific* (head 0, layer 0) -- with `n_heads=4,
  n_layers=2`, a different head or layer could easily look different; this
  notebook only checked the one already used throughout Part 3.

This matches the theory built earlier better than either extreme story
would: Graph 5 proved the *per-pair rotation* only ever depends on relative
distance, so it has no mathematical reason to break at position 129 --
and indeed nothing catastrophically breaks. The real, measurable cost
instead comes from the softmax now normalizing over more competing keys
than this model's weights ever adjusted for during training -- a real,
moderate, "graceful degradation," not the clean binary either extreme
story would predict.
